### Passo 1 - Importar Arquivos e Bibliotecas

In [ ]:
import pandas as pd
import pathlib
import win32com.client as win32

In [ ]:
email = pd.read_excel(r'C:\Users\tigor\OneDrive\Desktop\Projeto 1\Projeto AutomacaoIndicadores\Bases de Dados\Emails.xlsx')
lojas = pd.read_csv(r'C:\Users\tigor\OneDrive\Desktop\Projeto 1\Projeto AutomacaoIndicadores\Bases de Dados\Lojas.csv', encoding='latin1', sep=';')
vendas = pd.read_excel(r'C:\Users\tigor\OneDrive\Desktop\Projeto 1\Projeto AutomacaoIndicadores\Bases de Dados\Vendas.xlsx')

display(email)
display(lojas)
display(vendas)

### Passo 2 - Definir Criar uma Tabela para cada Loja e Definir o dia do Indicador

In [ ]:
vendas = vendas.merge(lojas, on='ID Loja')
display(vendas)



In [ ]:
dicionario_lojas = {}
for loja in lojas['Loja']:
    dicionario_lojas[loja] = vendas.loc[vendas['Loja']==loja, :]
display (dicionario_lojas['Shopping Vila Velha'])

In [ ]:
dia_indicador = vendas['Data'].max()
print('{}/{}'.format(dia_indicador.day,dia_indicador.month))

### Passo 3 - Salvar a planilha na pasta de backup

In [ ]:
caminho_salvar =pathlib.Path(r'C:\Users\tigor\OneDrive\Desktop\Projeto 1\Projeto AutomacaoIndicadores\Backup Arquivos Lojas')
arquivo_pasta_salva = caminho_salvar.iterdir()

lista_nome = [ arquivo.name for arquivo in arquivo_pasta_salva]

for loja in dicionario_lojas:
    if loja not in lista_nome:
        nova_pasta = caminho_salvar / loja
        nova_pasta.mkdir()
    nome_arquivo = '{}_{}_{}.xlsx'.format(dia_indicador.day,dia_indicador.month, loja)
    local_arquivo = caminho_salvar / loja / nome_arquivo
    dicionario_lojas[loja].to_excel(local_arquivo)


### Passo 4 - Calcular o indicador para 1 loja

In [ ]:
#metas
meta_faturamento_dia = 1000
meta_faturamento_ano = 1650000
meta_qtd_produto_dia = 4
meta_qtd_produto_ano = 120
meta_ticket_dia = 500
meta_ticket_ano = 500

In [ ]:
for loja in dicionario_lojas:
    
    vendas_loja = dicionario_lojas[loja]
    vendas_loja_dia = vendas_loja.loc[vendas_loja['Data']==dia_indicador, :]

    #faturamento
    faturamento_ano = vendas_loja['Valor Final'].sum()

    faturamento_dia = vendas_loja_dia['Valor Final'].sum()


    #diversidade
    qtd_produto_ano = len(vendas_loja['Produto'].unique())
    qtd_produto_dia = len(vendas_loja_dia['Produto'].unique())
    
    #ticket medio
    valor_venda_ano = vendas_loja.groupby('Código Venda')['Valor Final'].sum()
    ticket_ano = valor_venda_ano.mean()

    valor_venda_dia = vendas_loja_dia.groupby('Código Venda')['Valor Final'].sum()
    ticket_dia = valor_venda_dia.mean()

    
    #e-mail
    outlook = win32.Dispatch('outlook.application')

    nome = email.loc[email['Loja']==loja, 'Gerente'].values[0]
    mail = outlook.CreateItem(0)
    mail.To = email.loc[email['Loja']==loja, 'E-mail'].values[0]
    mail.Subject = 'OnePage Dia {}/{} - loja {}'.format(dia_indicador.day, dia_indicador.month, loja)

    # cores do cenário
    if faturamento_dia >= meta_faturamento_dia:
        cor_fat_dia = 'green'
    else:
        cor_fat_dia = 'red'
    if faturamento_ano >= meta_faturamento_ano:
        cor_fat_ano = 'green'
    else:
        cor_fat_ano = 'red'

    if qtd_produto_dia >= meta_qtd_produto_dia:
        cor_qtd_dia = 'green'
    else:
        cor_qtd_dia = 'red'
    if qtd_produto_ano >= meta_qtd_produto_ano:
        cor_qtd_ano = 'green'
    else:
        cor_qtd_ano = 'red'

    if ticket_dia >= meta_ticket_dia:
        cor_ticket_dia = 'green'
    else:
        cor_ticket_dia = 'red'
    if ticket_ano >= meta_ticket_ano:
        cor_ticket_ano = 'green'
    else:
        cor_ticket_ano = 'red'

    mail.HTMLBody = f'''
    <p>Bom dia, {nome}</p>

    <p>O resultado de ontem <strong>({dia_indicador.day}/{dia_indicador.month})</strong> da <strong>Loja {loja}</strong> foi:</p>

    <table>
        <tr>
        <th>Indicador</th>
        <th>Valor Dia</th>
        <th>Meta Dia</th>
        <th>Cenário Dia</th>
        </tr>
        <tr>
        <td>Faturamento</td>
        <td style="text-align: center">R${faturamento_dia:.2f}</td>
        <td style="text-align: center">R${meta_faturamento_dia:.2f}</td>
        <td style="text-align: center"><font color="{cor_fat_dia}">◙</font></td>
        </tr>
        <tr>
        <td>Diversidade de Produtos</td>
        <td style="text-align: center">{qtd_produto_dia}</td>
        <td style="text-align: center">{meta_qtd_produto_dia}</td>
        <td style="text-align: center"><font color="{cor_qtd_dia}">◙</font></td>
        </tr>
        <tr>
        <td>Ticket Médio</td>
        <td style="text-align: center">R${ticket_dia:.2f}</td>
        <td style="text-align: center">R${meta_ticket_dia:.2f}</td>
        <td style="text-align: center"><font color="{cor_ticket_dia}">◙</font></td>
        </tr>
    </table>
    <br>
    <table>
        <tr>
        <th>Indicador</th>
        <th>Valor Ano</th>
        <th>Meta Ano</th>
        <th>Cenário Ano</th>
        </tr>
        <tr>
        <td>Faturamento</td>
        <td style="text-align: center">R${faturamento_ano:.2f}</td>
        <td style="text-align: center">R${meta_faturamento_ano:.2f}</td>
        <td style="text-align: center"><font color="{cor_fat_ano}">◙</font></td>
        </tr>
        <tr>
        <td>Diversidade de Produtos</td>
        <td style="text-align: center">{qtd_produto_ano}</td>
        <td style="text-align: center">{meta_qtd_produto_ano}</td>
        <td style="text-align: center"><font color="{cor_qtd_ano}">◙</font></td>
        </tr>
        <tr>
        <td>Ticket Médio</td>
        <td style="text-align: center">R${ticket_ano:.2f}</td>
        <td style="text-align: center">R${meta_ticket_ano:.2f}</td>
        <td style="text-align: center"><font color="{cor_ticket_ano}">◙</font></td>
        </tr>
    </table>

    <p>Segue em anexo a planilha com todos os dados para mais detalhes.</p>

    <p>Qualquer dúvida estou à disposição.</p>
    <p>Att., Lira</p>
    '''

    #anexos
    attachment = caminho_salvar / loja / '{}_{}_{}.xlsx'.format(dia_indicador.day, dia_indicador.month, loja)
    mail.Attachments.Add(str(attachment))

    mail.Send()
    print('E-mail enviado para {}'.format(loja))



### Passo 7 - Criar ranking para diretoria

In [ ]:
faturamento_loja = vendas.groupby('Loja')[['Valor Final']].sum()
faturamento_loja_ano = faturamento_loja.sort_values(by='Valor Final', ascending=False)
display(faturamento_loja_ano)

nome_ano = '{}_{}_{}.Ranking anual.xlsx'.format(dia_indicador.day, dia_indicador.month, loja)
faturamento_loja_ano.to_excel(caminho_salvar / nome_ano)

vendas_dia = vendas.loc[vendas['Data'] == dia_indicador, :]
faturamento_loja_dia = vendas_dia.groupby('Loja')[['Valor Final']].sum()
faturamento_loja_dia = faturamento_loja_dia.sort_values(by='Valor Final', ascending=False)
display(faturamento_loja_dia)

nome_dia = '{}_{}_{}.Ranking Dia.xlsx'.format(dia_indicador.day, dia_indicador.month, loja)
faturamento_loja_dia.to_excel(caminho_salvar / nome_dia)

### Passo 8 - Enviar e-mail para diretoria

In [ ]:
print(nome_ano)
print((caminho_salvar / nome_ano).exists())
print([a.name for a in caminho_salvar.glob('*.xlsx')])

In [ ]:
# enviar o e-mail
outlook = win32.Dispatch('outlook.application')

mail = outlook.CreateItem(0)
mail.To = email.loc[email['Loja'] == 'Diretoria', 'E-mail'].values[0]
mail.Subject = f'Ranking Dia {dia_indicador.day}/{dia_indicador.month}'
mail.Body = f'''
Prezados, bom dia

Melhor loja do Dia em Faturamento: Loja {faturamento_loja_dia.index[0]} com Faturamento R${faturamento_loja_dia.iloc[0, 0]:.2f}
Pior loja do Dia em Faturamento: Loja {faturamento_loja_dia.index[-1]} com Faturamento R${faturamento_loja_dia.iloc[-1, 0]:.2f}

Melhor loja do Ano em Faturamento: Loja {faturamento_loja_ano.index[0]} com Faturamento R${faturamento_loja_ano.iloc[0, 0]:.2f}
Pior loja do Ano em Faturamento: Loja {faturamento_loja_ano.index[-1]} com Faturamento R${faturamento_loja_ano.iloc[-1, 0]:.2f}

Segue em anexo os rankings do ano e do dia de todas as lojas.

Qualquer dúvida estou à disposição.

Att.,
Lira
'''

# anexos
for anexo in [caminho_salvar / nome_ano, caminho_salvar / nome_dia]:
    if not anexo.exists():
        raise FileNotFoundError(f'Arquivo não encontrado: {anexo}')
    mail.Attachments.Add(str(anexo))

mail.Send()
print('E-mail da Diretoria enviado')